# Foreign Gifts Analysis

This notebook demonstrates various analytical capabilities for exploring foreign gifts data.

## Setup

First, let's import the necessary libraries and load our data.

In [ ]:
import pandas as pd
import sqlite_utils
from foreign_gifts.analysis.analyzer import GiftsAnalyzer
from foreign_gifts.analysis.classifier import GiftClassifier
from foreign_gifts.analysis.enrichment import DataEnricher
from foreign_gifts.analysis.visualizer import GiftsVisualizer

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data
db_path = "../data/gifts.db"
db = sqlite_utils.Database(db_path)

# Convert to pandas DataFrame
df = pd.DataFrame(db["gifts"].rows)

print(f"Loaded {len(df):,} gifts")
df.head()

## 1. Summary Statistics

Let's start by getting an overview of the dataset.

In [ ]:
analyzer = GiftsAnalyzer(db_path)
stats = analyzer.get_summary_statistics()

print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(f"Total Gifts: {stats['total_gifts']:,}")
print(f"Gifts with Value Data: {stats['gifts_with_value']:,} ({stats['gifts_with_value']/stats['total_gifts']*100:.1f}%)")
print(f"\nValue Statistics:")
print(f"  Total Value: ${stats['total_value']:,.2f}" if stats['total_value'] else "  Total Value: N/A")
print(f"  Average Value: ${stats['average_value']:,.2f}" if stats['average_value'] else "  Average Value: N/A")
print(f"  Min Value: ${stats['min_value']:,.2f}" if stats['min_value'] else "  Min Value: N/A")
print(f"  Max Value: ${stats['max_value']:,.2f}" if stats['max_value'] else "  Max Value: N/A")
print(f"\nUnique Donor Countries: {stats['unique_countries']}")
print(f"Unique Recipients: {stats['unique_recipients']}")

## 2. Top Donor Countries

Which countries give the most gifts?

In [ ]:
top_countries = analyzer.get_top_donor_countries(limit=15)

# Create DataFrame for better display
countries_df = pd.DataFrame(top_countries)
countries_df.index = range(1, len(countries_df) + 1)
countries_df.index.name = 'Rank'

print("\nTop 15 Donor Countries:")
countries_df[['country', 'gift_count', 'total_value', 'avg_value']]

## 3. Top Recipients

Who receives the most gifts?

In [ ]:
top_recipients = analyzer.get_top_recipients(limit=15)

recipients_df = pd.DataFrame(top_recipients)
recipients_df.index = range(1, len(recipients_df) + 1)
recipients_df.index.name = 'Rank'

print("\nTop 15 Recipients:")
recipients_df[['name', 'title', 'gift_count', 'total_value']]

## 4. Most Valuable Gifts

What are the most expensive gifts?

In [ ]:
valuable_gifts = analyzer.get_most_valuable_gifts(limit=10)

print("\n10 Most Valuable Gifts:")
print("=" * 100)

for i, gift in enumerate(valuable_gifts, 1):
    print(f"\n{i}. ${gift['estimated_value']:,.2f}")
    print(f"   Description: {gift['gift_description'][:80]}..." if len(gift['gift_description']) > 80 else f"   Description: {gift['gift_description']}")
    print(f"   From: {gift['donor_name']} ({gift['donor_country']})")
    print(f"   To: {gift['recipient_name']} - {gift['recipient_title']}")
    print(f"   Date: {gift['received']}")
    print(f"   Disposition: {gift['disposition']}")

## 5. Gift Categories

What types of gifts are most common?

In [ ]:
categories = analyzer.get_gift_categories()

categories_df = pd.DataFrame(list(categories.items()), columns=['Category', 'Count'])
categories_df = categories_df.sort_values('Count', ascending=False).head(20)

print("\nTop 20 Gift Categories:")
categories_df

## 6. Temporal Analysis

How have gifts changed over time?

In [ ]:
by_year = analyzer.get_gifts_by_year()

year_df = pd.DataFrame.from_dict(by_year, orient='index')
year_df.index.name = 'Year'
year_df = year_df.sort_index(ascending=False)

print("\nGifts by Year:")
year_df

## 7. Advanced Search

Let's search for specific types of gifts.

In [ ]:
# Search for jewelry
jewelry_gifts = analyzer.search_gifts(keyword="necklace")

print(f"\nFound {len(jewelry_gifts)} gifts with 'necklace' in description")
print("\nSample results:")

for i, gift in enumerate(jewelry_gifts[:5], 1):
    print(f"\n{i}. {gift['gift_description'][:80]}")
    print(f"   From: {gift['donor_country']} | To: {gift['recipient_name']}")
    if gift['estimated_value']:
        print(f"   Value: ${gift['estimated_value']:,.2f}")

In [ ]:
# Search for high-value gifts from specific country
expensive_saudi_gifts = analyzer.search_gifts(
    country="Saudi Arabia",
    min_value=5000
)

print(f"\nFound {len(expensive_saudi_gifts)} gifts from Saudi Arabia worth over $5,000")

for i, gift in enumerate(expensive_saudi_gifts[:5], 1):
    print(f"\n{i}. ${gift['estimated_value']:,.2f} - {gift['gift_description'][:60]}")
    print(f"   To: {gift['recipient_name']}")

## 8. Gift Classification

Let's use the classifier to categorize gifts automatically.

In [ ]:
classifier = GiftClassifier()

# Classify some sample gifts
sample_gifts = df.sample(10)[['gift_description', 'estimated_value']]

print("\nClassifying Sample Gifts:")
print("=" * 100)

for idx, row in sample_gifts.iterrows():
    desc = row['gift_description']
    classification = classifier.classify(desc)
    materials = classifier.extract_materials(desc)
    value_category = classifier.get_value_category(row['estimated_value'])
    
    print(f"\nDescription: {desc[:70]}..." if len(desc) > 70 else f"\nDescription: {desc}")
    print(f"Category: {classification['category_name']}")
    if classification['subcategory']:
        print(f"Subcategory: {classification['subcategory']}")
    print(f"Confidence: {classification['confidence']:.1%}")
    if materials:
        print(f"Materials: {', '.join(materials)}")
    print(f"Value Category: {value_category}")

## 9. Data Enrichment

Enrich gift data with additional metadata.

In [ ]:
enricher = DataEnricher()

# Enrich a sample gift
sample_gift = df.iloc[0].to_dict()
enriched = enricher.enrich_gift(sample_gift)

print("\nOriginal Gift Data:")
for key in ['gift_description', 'donor_country', 'estimated_value', 'received']:
    print(f"{key}: {sample_gift.get(key)}")

print("\nEnriched Fields:")
enriched_keys = [k for k in enriched.keys() if k not in sample_gift]
for key in enriched_keys:
    print(f"{key}: {enriched[key]}")

## 10. Visualizations

Create charts and graphs to visualize the data.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

visualizer = GiftsVisualizer(db_path)

# Create visualizations
print("Generating visualizations...\n")

print("1. Top Donor Countries")
visualizer.plot_top_donor_countries(limit=15)
plt.show()

print("\n2. Value Distribution")
visualizer.plot_value_distribution()
plt.show()

print("\n3. Gifts Over Time")
visualizer.plot_gifts_over_time()
plt.show()

print("\n4. Regional Distribution")
visualizer.plot_regional_distribution()
plt.show()

print("\n5. Top Recipients")
visualizer.plot_recipient_analysis(limit=12)
plt.show()

## 11. Custom Analysis Examples

### Example 1: Average Gift Value by Country

In [ ]:
# Countries with highest average gift values
query = """
    SELECT
        donor_country,
        COUNT(*) as gift_count,
        AVG(estimated_value) as avg_value,
        MAX(estimated_value) as max_value
    FROM gifts
    WHERE estimated_value IS NOT NULL
    AND donor_country IS NOT NULL
    GROUP BY donor_country
    HAVING gift_count >= 5
    ORDER BY avg_value DESC
    LIMIT 15
"""

avg_by_country = pd.read_sql_query(query, db.conn)
print("\nCountries with Highest Average Gift Values (min 5 gifts):")
avg_by_country

### Example 2: Gift Disposition Analysis

In [ ]:
disposition_stats = analyzer.get_disposition_analysis()

disp_df = pd.DataFrame(disposition_stats['disposition_breakdown'])
print("\nGift Disposition Analysis:")
disp_df

### Example 3: Regional Comparison

In [ ]:
# Enrich all gifts with regional data
enriched_gifts = [enricher.enrich_gift(gift) for gift in df.to_dict('records')]
regional_stats = enricher.get_regional_statistics(enriched_gifts)

print("\nGifts by Region:")
print("=" * 70)

for region, stats in sorted(regional_stats.items(), key=lambda x: x[1]['count'], reverse=True):
    print(f"\n{region}:")
    print(f"  Gifts: {stats['count']}")
    print(f"  Total Value: ${stats['total_value']:,.2f}")
    print(f"  Average Value: ${stats['avg_value']:,.2f}")

## 12. Generate Full Report

Create a comprehensive text report.

In [ ]:
report = analyzer.generate_report()
print(report)

## Conclusion

This notebook demonstrated:

1. **Summary Statistics** - Overall dataset overview
2. **Top Donors & Recipients** - Who gives and receives most
3. **Value Analysis** - Most expensive gifts and distributions
4. **Categorization** - Automated gift classification
5. **Temporal Trends** - How gifts change over time
6. **Search Capabilities** - Find specific gifts
7. **Data Enrichment** - Add metadata and context
8. **Visualizations** - Charts and graphs
9. **Custom Analysis** - Flexible querying

You can use these techniques to explore the data and answer specific questions about foreign gift-giving patterns.